# Notebook 18 – Model Comparison

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.


## Setup: Load Data & Train Models
Load data, engineer `TotalPrice`, split into train/validation/test, and train all 3 models so every topic below can use them.

In [16]:
import pandas as pd, numpy as np, time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(5000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8)
}
train_times = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    train_times[name] = time.time() - start
print("Models trained:", list(models.keys()))

Models trained: ['Logistic Regression', 'Decision Tree', 'Random Forest']


## 1. Algorithm
Just the name of each model we're comparing — this is the first column of our final table.

In [2]:
algorithms = list(models.keys())
algorithms

['Logistic Regression', 'Decision Tree', 'Random Forest']

## 2. Training Score
Accuracy on the data the model was **trained on**. High training score alone doesn't mean much — a model can memorize training data.

In [3]:
train_scores = {name: accuracy_score(y_train, m.predict(X_train)) for name, m in models.items()}
train_scores

{'Logistic Regression': 0.8906666666666667,
 'Decision Tree': 0.8973333333333333,
 'Random Forest': 0.8963333333333333}

## 3. Validation Score
Accuracy on a **held-out validation set** the model never trained on. Used to tune choices without touching the final test set.

In [4]:
val_scores = {name: accuracy_score(y_val, m.predict(X_val)) for name, m in models.items()}
val_scores

{'Logistic Regression': 0.889, 'Decision Tree': 0.887, 'Random Forest': 0.89}

## 4. Test Score
Accuracy on the **final, untouched test set** — the truest estimate of real-world performance.

In [5]:
test_scores = {name: accuracy_score(y_test, m.predict(X_test)) for name, m in models.items()}
test_scores

{'Logistic Regression': 0.889, 'Decision Tree': 0.895, 'Random Forest': 0.894}

## 5. Accuracy / R²
Since this is a **classification** task (not regression), we use **Accuracy** (fraction of correct predictions). R² would only apply if we were predicting a number instead.

In [7]:
accuracy = test_scores  
accuracy

{'Logistic Regression': 0.889, 'Decision Tree': 0.895, 'Random Forest': 0.894}

## 6. Precision
Of all orders the model predicted as "UK", what fraction were actually UK? High precision means fewer false alarms.

In [8]:
precision = {name: precision_score(y_test, m.predict(X_test)) for name, m in models.items()}
precision

{'Logistic Regression': 0.8907815631262525,
 'Decision Tree': 0.8985801217038539,
 'Random Forest': 0.8952668680765358}

## 7. Recall
Of all the actual UK orders, what fraction did the model correctly catch? High recall means fewer missed UK orders.

In [9]:
recall = {name: recall_score(y_test, m.predict(X_test)) for name, m in models.items()}
recall

{'Logistic Regression': 0.9977553310886644,
 'Decision Tree': 0.9943883277216611,
 'Random Forest': 0.9977553310886644}

## 8. F1 Score
The balance between Precision and Recall (harmonic mean). Useful when both false alarms and missed cases matter.

In [10]:
f1 = {name: f1_score(y_test, m.predict(X_test)) for name, m in models.items()}
f1

{'Logistic Regression': 0.9412387506617258,
 'Decision Tree': 0.9440596696856686,
 'Random Forest': 0.9437367303609342}

## 9. MAE / RMSE
**Not applicable here** — MAE (Mean Absolute Error) and RMSE (Root Mean Squared Error) are regression metrics for predicting numbers. Our task is classification, so we mark these as N/A.

In [11]:
mae_rmse = {name: 'N/A (classification task)' for name in models}
mae_rmse

{'Logistic Regression': 'N/A (classification task)',
 'Decision Tree': 'N/A (classification task)',
 'Random Forest': 'N/A (classification task)'}

## 10. Training Time
How long each model took to train. Simpler models are usually much faster — relevant if you need to retrain often.

In [12]:
train_times

{'Logistic Regression': 0.016259431838989258,
 'Decision Tree': 0.010672807693481445,
 'Random Forest': 0.3515207767486572}

## 11. Model Complexity
A simple description of how complex each model is internally (number of coefficients, tree depth, number of trees).

In [13]:
complexity = {
    'Logistic Regression': "Low (3 coefficients, linear)",
    'Decision Tree': f"Medium (depth={models['Decision Tree'].get_depth()}, {models['Decision Tree'].get_n_leaves()} leaves)",
    'Random Forest': f"High ({models['Random Forest'].n_estimators} trees, max_depth={models['Random Forest'].max_depth})"
}
complexity

{'Logistic Regression': 'Low (3 coefficients, linear)',
 'Decision Tree': 'Medium (depth=6, 38 leaves)',
 'Random Forest': 'High (100 trees, max_depth=8)'}

## 12. Full Comparison Table
Putting every metric above together into one table.

In [14]:
comparison_table = pd.DataFrame({
    'Algorithm': algorithms,
    'Training Score': [round(train_scores[a], 3) for a in algorithms],
    'Validation Score': [round(val_scores[a], 3) for a in algorithms],
    'Test Score': [round(test_scores[a], 3) for a in algorithms],
    'Accuracy': [round(accuracy[a], 3) for a in algorithms],
    'Precision': [round(precision[a], 3) for a in algorithms],
    'Recall': [round(recall[a], 3) for a in algorithms],
    'F1': [round(f1[a], 3) for a in algorithms],
    'MAE / RMSE': [mae_rmse[a] for a in algorithms],
    'Training Time (s)': [round(train_times[a], 4) for a in algorithms],
    'Model Complexity': [complexity[a] for a in algorithms],
})
comparison_table

,Algorithm,Training Score,Validation Score,Test Score,Accuracy,Precision,Recall,F1,MAE / RMSE,Training Time (s),Model Complexity
0,Logistic Regression,0.891,0.889,0.889,0.889,0.891,0.998,0.941,N/A (classification task),0.0163,"Low (3 coefficients, linear)"
1,Decision Tree,0.897,0.887,0.895,0.895,0.899,0.994,0.944,N/A (classification task),0.0107,"Medium (depth=6, 38 leaves)"
2,Random Forest,0.896,0.890,0.894,0.894,0.895,0.998,0.944,N/A (classification task),0.3515,"High (100 trees, max_depth=8)"


## 13. Model Selection — Why Not Just "Highest Accuracy"?

We look at more than one number before choosing:
- **Train-Test Gap** → a big gap means overfitting, even if test accuracy looks fine.
- **Precision/Recall/F1** → accuracy alone can hide a model that fails on the minority class.
- **Complexity & Training Time** → a small accuracy gain from a much more complex model often isn't worth it.

**Selected model: Logistic Regression** — it has the smallest train-test gap (best generalization), is far faster to train, and is easiest to interpret, even if a more complex model shows marginally higher raw accuracy.

In [15]:
comparison_table['Train-Test Gap'] = (comparison_table['Training Score'] - comparison_table['Test Score']).abs()
best = comparison_table.loc[comparison_table['Train-Test Gap'].idxmin()]
print("Best generalizing model:")
print(best[['Algorithm', 'Training Score', 'Test Score', 'Train-Test Gap']])

Best generalizing model:
Algorithm         Logistic Regression
Training Score                  0.891
Test Score                      0.889
Train-Test Gap                  0.002
Name: 0, dtype: object
